# Domain 4: Prompt Engineering & Structured Output
## Claude Certified Architect – Foundations Certification
**Weight: 20% of scored content**

This notebook covers designing precise prompts, few-shot prompting, enforcing structured output
via tool use and JSON schemas, validation-retry loops, batch processing, and multi-pass review.

---

## Task 4.1: Design Prompts with Explicit Criteria to Improve Precision and Reduce False Positives

### The Problem with Vague Instructions

| Instruction | Effectiveness |
|---|---|
| ❌ "Be conservative" | Fails to improve precision |
| ❌ "Only report high-confidence findings" | Vague — model can't calibrate |
| ❌ "Check that comments are accurate" | Too broad |
| ✅ "Flag comments only when claimed behavior contradicts actual code behavior" | Specific and actionable |

### Impact of False Positives

High false positive rates in specific categories **undermine developer trust** across ALL categories,
even accurate ones.

### Strategies

1. Define **specific review criteria** (what to report vs skip)
   - Report: bugs, security vulnerabilities
   - Skip: minor style issues, local patterns
2. **Temporarily disable** high false-positive categories while improving prompts
3. Define **severity criteria with concrete code examples** for each level

In [ ]:
# === EXPLICIT CRITERIA DESIGN ===

# ❌ BAD: Vague prompt
vague_prompt = """
Review this code for issues. Be conservative and only report 
high-confidence findings.
"""

# ✅ GOOD: Explicit criteria with concrete examples
explicit_prompt = """
Review this code. Report ONLY the following issue types:

## REPORT (bugs, security, logic errors):
- Null pointer dereferences where a variable could be None/null
- SQL injection: string concatenation in SQL queries
- Race conditions in concurrent code without proper locking
- Off-by-one errors in loop boundaries or array access

## SKIP (do not report):
- Code style preferences (naming, formatting)
- Minor optimization opportunities
- Missing comments or documentation
- Patterns that are project conventions (even if unusual)

## SEVERITY DEFINITIONS:
- CRITICAL: Data loss, security breach, or system crash
  Example: `query = f"SELECT * FROM users WHERE id = {user_input}"` (SQL injection)
- HIGH: Incorrect behavior under normal usage
  Example: `if len(items) > 0` when items could be None (AttributeError)
- MEDIUM: Incorrect behavior under edge cases
  Example: `for i in range(len(arr) - 1)` missing the last element
- LOW: Potential issue requiring specific conditions
  Example: Race condition only triggered under high concurrency
"""

print("=== Vague Prompt (BAD) ===")
print(vague_prompt.strip())
print("\n=== Explicit Criteria Prompt (GOOD) ===")
print(explicit_prompt.strip()[:300] + "...")

---

## Task 4.2: Apply Few-Shot Prompting to Improve Output Consistency and Quality

### Why Few-Shot Examples Are Powerful

- **Most effective technique** for consistent, formatted, actionable output
- Enable the model to **generalize** to novel patterns (not just match pre-specified cases)
- Demonstrate ambiguous-case handling and reasoning
- Reduce hallucination in extraction tasks

### Best Practices

- Provide **2-4 targeted examples** for ambiguous scenarios
- Show **reasoning** for why one action was chosen over alternatives
- Include examples that distinguish acceptable patterns from genuine issues
- Demonstrate correct handling of **varied document structures**

In [ ]:
# === FEW-SHOT PROMPTING EXAMPLES ===

few_shot_review_prompt = """
Review code changes and output findings in this exact format:

## Example 1 (report — real bug):
Input code:
```python
def get_user(user_id):
    user = db.find(user_id)
    return user.name  # Could be None if user not found
```
Output:
{"file": "users.py", "line": 3, "severity": "high", 
 "issue": "Potential AttributeError: db.find() may return None",
 "suggestion": "Add null check: if user is None: return None"}

## Example 2 (skip — acceptable pattern):
Input code:
```python
# Using list comprehension for filtering (project convention)
active = [u for u in users if u.active]
```
Output: No finding (this is an acceptable project pattern)
Reasoning: List comprehension for filtering is a valid Python pattern, 
not a bug. Don't flag code style preferences.

## Example 3 (report — security issue):
Input code:
```python
query = f"SELECT * FROM orders WHERE id = {order_id}"
```
Output:
{"file": "orders.py", "line": 1, "severity": "critical",
 "issue": "SQL injection vulnerability: user input in f-string query",
 "suggestion": "Use parameterized query: cursor.execute('SELECT * FROM orders WHERE id = %s', (order_id,))"}

## Example 4 (ambiguous — shows reasoning):
Input code:
```python
timeout = 30  # hardcoded timeout
```
Output: No finding
Reasoning: While hardcoded values could be a concern, a timeout of 30s 
is a reasonable default. Only flag hardcoded values that represent 
security risks (passwords, API keys) or values that clearly should be 
configurable based on context.
"""

print("Few-shot prompt demonstrates:")
print("  1. Exact output format (JSON structure)")
print("  2. What to report (real bugs, security issues)")
print("  3. What to skip (acceptable patterns)")
print("  4. Reasoning for ambiguous cases")
print("  5. How to generalize to novel similar cases")

---

## Task 4.3: Enforce Structured Output Using Tool Use and JSON Schemas

### The Most Reliable Approach

**`tool_use` with JSON schemas** = guaranteed schema-compliant output, eliminating JSON syntax errors.

### `tool_choice` Options

| Setting | Behavior | Use When |
|---|---|---|
| `"auto"` | Model may return text instead of calling a tool | General conversation |
| `"any"` | Model MUST call a tool (can choose which) | Multiple extraction schemas, unknown doc type |
| `{"type": "tool", "name": "..."}` | Forces a specific named tool | Ensure specific extraction runs first |

### Important Distinctions

- **Strict JSON schemas eliminate syntax errors** but NOT semantic errors
  - ✅ Prevents malformed JSON
  - ❌ Cannot prevent: line items that don't sum to total, values in wrong fields

### Schema Design Tips

- Make fields **optional (nullable)** when source docs may not contain the info
  - Prevents model from **fabricating values** to satisfy required fields
- Use **enum + "other" + detail string** pattern for extensible categories
- Include format normalization rules alongside schemas

In [ ]:
# === STRUCTURED OUTPUT WITH TOOL USE ===

import json

# Define extraction tool with JSON schema
extraction_tool = {
    "name": "extract_invoice_data",
    "description": (
        "Extract structured data from an invoice document. "
        "Use this tool to return the extracted fields in a consistent format."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "invoice_number": {
                "type": ["string", "null"],  # Nullable — may not exist
                "description": "Invoice number/ID"
            },
            "vendor_name": {
                "type": "string",
                "description": "Name of the vendor/supplier"
            },
            "total_amount": {
                "type": "number",
                "description": "Total invoice amount"
            },
            "calculated_total": {
                "type": ["number", "null"],
                "description": "Sum of line items (for validation against stated total)"
            },
            "currency": {
                "type": "string",
                "enum": ["USD", "EUR", "GBP", "JPY", "other"],
                "description": "Currency code"
            },
            "currency_detail": {
                "type": ["string", "null"],
                "description": "Specify currency when 'other' is selected"
            },
            "date": {
                "type": ["string", "null"],
                "description": "Invoice date in ISO 8601 format (YYYY-MM-DD)"
            },
            "payment_status": {
                "type": "string",
                "enum": ["paid", "unpaid", "partial", "overdue", "unclear"],
                "description": "Payment status. Use 'unclear' when status is ambiguous."
            },
            "line_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "quantity": {"type": "number"},
                        "unit_price": {"type": "number"},
                        "total": {"type": "number"}
                    }
                }
            },
            "conflict_detected": {
                "type": "boolean",
                "description": "True if calculated_total differs from total_amount"
            }
        },
        "required": ["vendor_name", "total_amount", "currency", "payment_status"]
    }
}

print("Extraction Tool Schema Design:")
print(f"  Tool: {extraction_tool['name']}")
props = extraction_tool['input_schema']['properties']
required = extraction_tool['input_schema']['required']
for name, schema in props.items():
    nullable = isinstance(schema.get('type'), list) and 'null' in schema['type']
    req = '✅ Required' if name in required else '⬜ Optional'
    null_str = ' (nullable)' if nullable else ''
    print(f"  {req} {name}{null_str}")

print("\nKey design patterns:")
print("  - Nullable fields prevent fabrication when info is absent")
print("  - Enum with 'unclear' for ambiguous cases")
print("  - Enum with 'other' + detail field for extensible categories")
print("  - calculated_total + conflict_detected for self-validation")

---

## Task 4.4: Implement Validation, Retry, and Feedback Loops for Extraction Quality

### Retry-with-Error-Feedback Pattern

When extraction validation fails:
1. Append the **original document**
2. Append the **failed extraction**
3. Append **specific validation errors**
4. Ask for correction

### When Retries Are Effective vs Ineffective

| Scenario | Retry Effective? | Reason |
|---|---|---|
| Format mismatches (date formats) | ✅ Yes | Model can fix formatting |
| Structural output errors | ✅ Yes | Model can restructure |
| Info only in external document | ❌ No | Model can't access what's not provided |
| Info absent from source | ❌ No | No amount of retrying creates missing data |

### Feedback Loop Design

Add `detected_pattern` field to findings → enables systematic analysis of dismissal patterns

In [ ]:
# === VALIDATION-RETRY LOOP ===

from dataclasses import dataclass, field
from typing import Optional

@dataclass
class ValidationError:
    field: str
    error_type: str  # 'format', 'missing', 'inconsistent', 'absent_from_source'
    message: str
    retryable: bool

def validate_extraction(extraction: dict) -> list:
    """Validate extracted data and return specific errors."""
    errors = []
    
    # Check date format
    if extraction.get('date') and not extraction['date'].startswith('20'):
        errors.append(ValidationError(
            field='date',
            error_type='format',
            message=f"Date '{extraction['date']}' is not ISO 8601 (expected YYYY-MM-DD)",
            retryable=True  # Model can fix formatting
        ))
    
    # Check calculated vs stated total
    calc = extraction.get('calculated_total')
    stated = extraction.get('total_amount')
    if calc and stated and abs(calc - stated) > 0.01:
        errors.append(ValidationError(
            field='total_amount',
            error_type='inconsistent',
            message=f"Calculated total ({calc}) differs from stated total ({stated})",
            retryable=True  # Model can re-examine
        ))
    
    # Check for fabricated values (required field but source may not have it)
    if extraction.get('invoice_number') == 'N/A':
        errors.append(ValidationError(
            field='invoice_number',
            error_type='absent_from_source',
            message="Invoice number not found in source — return null instead of 'N/A'",
            retryable=True  # Model can return null
        ))
    
    return errors

def build_retry_prompt(original_doc, failed_extraction, errors):
    """
    Retry-with-error-feedback pattern.
    Include: original document + failed extraction + specific errors.
    """
    retryable_errors = [e for e in errors if e.retryable]
    
    if not retryable_errors:
        return None  # Don't retry — info is absent from source
    
    error_details = "\n".join(
        f"- {e.field}: {e.message}" for e in retryable_errors
    )
    
    return f"""
The previous extraction had validation errors. Please correct them.

## Original Document
{original_doc}

## Previous Extraction (with errors)
{json.dumps(failed_extraction, indent=2)}

## Specific Errors to Fix
{error_details}

Please re-extract with these corrections.
"""

# Demo
test_extraction = {
    "vendor_name": "Acme Corp",
    "total_amount": 1500.00,
    "calculated_total": 1450.00,  # Mismatch!
    "date": "03/15/2024",  # Wrong format!
    "invoice_number": "N/A",  # Should be null
    "currency": "USD",
    "payment_status": "unpaid"
}

errors = validate_extraction(test_extraction)
print("Validation Errors Found:")
for e in errors:
    print(f"  [{e.error_type}] {e.field}: {e.message} (retryable: {e.retryable})")

retry = build_retry_prompt("<original doc>", test_extraction, errors)
if retry:
    print(f"\nRetry prompt generated ({len(retry)} chars)")
    print("  Includes: original doc + failed extraction + specific errors")

---

## Task 4.5: Design Efficient Batch Processing Strategies

### Message Batches API

| Feature | Detail |
|---|---|
| **Cost savings** | 50% reduction |
| **Processing window** | Up to 24 hours |
| **Latency SLA** | None guaranteed |
| **Multi-turn tool calling** | ❌ Not supported within a single batch request |
| **Request correlation** | `custom_id` fields |

### When to Use Batch vs Real-Time

| Workflow | API Choice | Reason |
|---|---|---|
| Pre-merge checks (blocking) | **Real-time** | Developers waiting — need immediate response |
| Overnight tech debt reports | **Batch** | Latency-tolerant, cost-sensitive |
| Weekly audits | **Batch** | Non-blocking, large volume |
| Nightly test generation | **Batch** | Can wait for results |

### Failure Handling

- Identify failed documents by `custom_id`
- Resubmit **only** failed items with modifications (e.g., chunking oversized docs)
- Refine prompts on a **sample set** before batch-processing large volumes

In [ ]:
# === BATCH PROCESSING STRATEGY ===

import json
from datetime import datetime, timedelta

def create_batch_request(documents):
    """Create batch API requests with custom_id for correlation."""
    requests = []
    for doc in documents:
        requests.append({
            "custom_id": doc["id"],  # For correlating responses
            "params": {
                "model": "claude-sonnet-4-20250514",
                "max_tokens": 1024,
                "messages": [
                    {"role": "user", "content": f"Extract data from: {doc['content']}"}
                ]
            }
        })
    return requests

def calculate_batch_timing(sla_hours, batch_processing_max=24):
    """
    Calculate batch submission frequency.
    Exam scenario: If SLA = 30 hours and batch takes up to 24 hours,
    submit every 4 hours (30 - 24 = 6 hour buffer, submit every 4 for safety).
    """
    buffer = sla_hours - batch_processing_max
    safe_frequency = max(1, buffer - 2)  # Safety margin
    return {
        "sla_hours": sla_hours,
        "max_processing": batch_processing_max,
        "buffer_hours": buffer,
        "submit_every_hours": safe_frequency
    }

def handle_batch_failures(results):
    """Resubmit only failed items with appropriate modifications."""
    failed = [r for r in results if r.get('error')]
    resubmit = []
    
    for item in failed:
        if 'context_limit' in str(item.get('error', '')):
            # Chunk oversized documents
            resubmit.append({
                "custom_id": item['custom_id'],
                "modification": "chunked",
                "reason": "Document exceeded context limit"
            })
        else:
            resubmit.append({
                "custom_id": item['custom_id'],
                "modification": "none",
                "reason": str(item.get('error'))
            })
    
    return resubmit

# Demo
timing = calculate_batch_timing(sla_hours=30)
print("Batch Timing Calculation:")
for k, v in timing.items():
    print(f"  {k}: {v}")

print("\nBatch vs Real-time Decision:")
print("  Pre-merge checks → REAL-TIME (blocking workflow)")
print("  Overnight reports → BATCH (50% cost savings)")
print("  Weekly audits → BATCH (latency-tolerant)")

---

## Task 4.6: Design Multi-Instance and Multi-Pass Review Architectures

### Self-Review Limitations

A model retains reasoning context from generation, making it **less likely to question
its own decisions** in the same session.

→ Use an **independent review instance** (separate session, no prior reasoning context)

### Multi-Pass Review Pattern

For large multi-file reviews (e.g., 14 files):

```
Pass 1: Per-file local analysis    → Consistent depth per file
Pass 2: Cross-file integration     → Data flow, API consistency
Pass 3: Confidence self-reporting  → Route findings by confidence
```

### Why This Matters

Single-pass reviews of many files produce:
- Inconsistent depth (detailed on some files, superficial on others)
- Contradictory feedback (flagging pattern in one file, approving same in another)
- Missed obvious bugs due to attention dilution

In [ ]:
# === MULTI-PASS REVIEW ARCHITECTURE ===

class MultiPassReviewer:
    """
    Split large reviews into focused passes.
    Addresses attention dilution in single-pass reviews.
    """
    
    def __init__(self, files):
        self.files = files
        self.local_findings = {}
        self.integration_findings = []
    
    def pass_1_local_analysis(self):
        """Analyze each file individually for local issues."""
        for f in self.files:
            # Each file gets a dedicated, focused review pass
            self.local_findings[f] = {
                "bugs": [],
                "security": [],
                "style": [],
                "confidence": None
            }
        return f"Pass 1: Analyzed {len(self.files)} files individually"
    
    def pass_2_integration_analysis(self):
        """Examine cross-file data flow and consistency."""
        # Looks at: API contracts, data flow, shared state, consistency
        self.integration_findings = [
            "Check API contract consistency across endpoints",
            "Verify data flow between modules",
            "Check for contradictory patterns across files"
        ]
        return f"Pass 2: Cross-file integration analysis"
    
    def pass_3_confidence_routing(self):
        """Self-report confidence for review routing."""
        # Model reports confidence alongside each finding
        # High confidence → auto-post as comment
        # Low confidence → route to human reviewer
        return "Pass 3: Confidence scoring for review routing"

# Demo
pr_files = [f"file_{i}.py" for i in range(14)]
reviewer = MultiPassReviewer(pr_files)

print(reviewer.pass_1_local_analysis())
print(reviewer.pass_2_integration_analysis())
print(reviewer.pass_3_confidence_routing())

print("\nKey insight: Independent review instance catches more")
print("issues than self-review in the same session.")

---

## Domain 4 Summary & Exam Preparation

### Key Concepts to Remember

1. **Explicit criteria** > vague instructions ("be conservative" doesn't work)
2. **Few-shot examples** are the most effective technique for consistent output
3. **`tool_use` with JSON schemas** eliminates syntax errors (but not semantic errors)
4. **Nullable fields** prevent fabrication when info is absent
5. **`tool_choice: "any"`** guarantees the model calls a tool
6. **Retry-with-error-feedback**: Include original doc + failed extraction + specific errors
7. **Don't retry** when info is absent from source (retries won't create missing data)
8. **Batch API**: 50% savings, up to 24h processing, no multi-turn tool calling
9. **Pre-merge checks** need real-time API (blocking); reports can use batch
10. **Independent review instance** > self-review (no shared reasoning context)
11. **Multi-pass reviews** prevent attention dilution in large PRs

### Common Exam Trap

> "Switch both pre-merge checks AND overnight reports to Batch API"
>
> ❌ Wrong — Batch API has no latency SLA, unsuitable for blocking workflows
>
> ✅ Only switch overnight reports; keep real-time for pre-merge checks